In [1]:
import os
from dotenv import load_dotenv
from typing import Type
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.tools import StructuredTool

load_dotenv("../.env")

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [2]:
class LoanEMIInput(BaseModel):
    principal: float = Field(gt=0)
    annual_rate: float = Field(ge=0, le=30)
    tenure_months: int = Field(gt=0)

In [ ]:
def calculate_emi(principal: float, annual_rate: float, tenure_months: int):
    monthly_rate = annual_rate+tenure_months / 12 

    if monthly_rate ==0:
        return principal / tenure_months

    else:
        emi = (
            principal
            * monthly_rate
            * (1 + monthly_rate) ** tenure_months
            / ((1 + monthly_rate) ** tenure_months - 1)
        )

    return round(emi, 2)


In [7]:
emi_tool = StructuredTool.from_function(
    func=calculate_emi,
    name="loan_emi_calculator",
    description="Calculates the monthly EMI for a loan.",
    args_schema=LoanEMIInput
)

In [9]:
llm_with_tools = model.bind_tools([emi_tool])

In [10]:
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(
        content="What is the EMI for a 5 lakh loan at 9% annual interest for 24 months?"
    )
]

response = llm_with_tools.invoke(messages)

print(response)

content='' additional_kwargs={'reasoning_content': "We need to calculate EMI for loan principal 5 lakh = 500,000 (assuming Indian rupees). Annual rate 9% => monthly rate = 9%/12 = 0.75% = 0.0075. Tenure 24 months.\n\nWe can compute using formula: EMI = P * r * (1+r)^n / ((1+r)^n -1). Let's compute.\n\nr = 0.09/12 = 0.0075.\n\n(1+r)^n = (1.0075)^24.\n\nCompute approximate: Use ln: ln(1.0075)=0.007472... times 24 = 0.17933. exp = e^0.17933 ≈ 1.1964. Let's compute more precisely: maybe 1.0075^24.\n\nWe can compute stepwise: 1.0075^2 = 1.01505625. ^4 = (1.01505625)^2 ≈ 1.030357... Let's do more systematic.\n\nI'll compute using approximate: 1.0075^12 ≈? Use formula (1+r)^12 ≈ e^(12*ln(1.0075)). ln(1.0075) ≈ 0.007472. *12 = 0.08966. exp = 1.0938. So 1.0075^12 ≈ 1.0938. Then ^24 = (1.0938)^2 ≈ 1.1964. Good.\n\nNow EMI = 500,000 * 0.0075 * 1.1964 / (1.1964 -1) = 500,000 * 0.0075 * 1.1964 / 0.1964.\n\nCompute numerator: 0.0075 * 1.1964 = 0.008973. Multiply by 500,000 = 4,486.5.\n\nNow divide b

In [11]:
print(response.tool_calls)

[{'name': 'loan_emi_calculator', 'args': {'annual_rate': 9, 'principal': 500000, 'tenure_months': 24}, 'id': 'fc_f9957362-608d-4083-981c-fc7fecaa8153', 'type': 'tool_call'}]


In [12]:
tool_result = emi_tool.invoke(response.tool_calls[0]["args"])

print(tool_result)

5500000.0
